In [ ]:

import torch
import torchvision
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

PyTorch version: 2.11.0+cpu
Torchvision version: 0.26.0+cpu


In [ ]:

from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 497kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.64MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.25MB/s]

Training samples: 60000
Test samples: 10000


In [ ]:

import torch

torch.manual_seed(42)

forget_size = 500

forget_indices = torch.randperm(len(train_dataset))[:forget_size]
retain_indices = torch.tensor([
    i for i in range(len(train_dataset))
    if i not in set(forget_indices.tolist())
])

print("Forget set:", len(forget_indices))
print("Retain set:", len(retain_indices))

Forget set: 500
Retain set: 59500


In [ ]:

from torch.utils.data import DataLoader, Subset

full_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

forget_loader = DataLoader(
    Subset(train_dataset, forget_indices),
    batch_size=128,
    shuffle=False
)

retain_loader = DataLoader(
    Subset(train_dataset, retain_indices),
    batch_size=128,
    shuffle=True
)

print("DataLoaders created successfully.")

DataLoaders created successfully.


In [ ]:

import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


model = SimpleMLP()

print(model)

SimpleMLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [ ]:

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in full_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {total_loss / len(full_loader):.4f}"
    )

Epoch 1/3, Loss: 1.7117
Epoch 2/3, Loss: 0.7741
Epoch 3/3, Loss: 0.5339


In [ ]:

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

for images, labels in full_loader:
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print("Full-model training complete.")

Full-model training complete.


In [ ]:

torch.save(model.state_dict(), "full_model.pth")
print("Model weights saved.")

Model weights saved.


In [ ]:

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/full_model.pth"
)

print("Model saved permanently to Google Drive.")

Model saved permanently to Google Drive.


In [ ]:

retrained_model = SimpleMLP()

retrain_optimizer = optim.SGD(
    retrained_model.parameters(),
    lr=0.01
)

retrained_model.train()

for images, labels in retain_loader:
    retrain_optimizer.zero_grad()

    outputs = retrained_model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    retrain_optimizer.step()

print("Retraining without forgotten data complete.")

Retraining without forgotten data complete.


In [ ]:

torch.save(
    retrained_model.state_dict(),
    "/content/drive/MyDrive/retrained_model.pth"
)

print("Retrained model saved.")

Retrained model saved.


In [ ]:

unlearned_model = SimpleMLP()

unlearned_model.load_state_dict(
    model.state_dict()
)

print("Unlearning model initialized from full model.")

Unlearning model initialized from full model.


In [ ]:

unlearned_model.eval()

unlearned_model.zero_grad()

forget_loss = 0.0
num_forget = 0

for images, labels in forget_loader:
    outputs = unlearned_model(images)
    loss = criterion(outputs, labels)

    forget_loss += loss * images.size(0)
    num_forget += images.size(0)

forget_loss = forget_loss / num_forget

forget_loss.backward()

print("Forgotten-data loss:", forget_loss.item())

grad_norm = torch.sqrt(
    sum(
        (p.grad.detach() ** 2).sum()
        for p in unlearned_model.parameters()
        if p.grad is not None
    )
)

print("Forgotten-data gradient norm:", grad_norm.item())

Forgotten-data loss: 1.0361506938934326
Forgotten-data gradient norm: 0.5467761158943176


In [ ]:
import torch

In [ ]:

import torch
import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


unlearned_model = SimpleMLP()

unlearned_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/full_model.pth",
        weights_only=True
    )
)

print("Full model restored for unlearning.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/full_model.pth'

In [ ]:

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os

print(os.path.exists("/content/drive/MyDrive/full_model.pth"))

True


In [ ]:

unlearned_model = SimpleMLP()

unlearned_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/full_model.pth",
        weights_only=True
    )
)

print("Full model restored for unlearning.")

Full model restored for unlearning.


In [ ]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

torch.manual_seed(42)

forget_size = 500
forget_indices = torch.randperm(len(train_dataset))[:forget_size]

forget_loader = DataLoader(
    Subset(train_dataset, forget_indices),
    batch_size=128,
    shuffle=False
)

criterion = nn.CrossEntropyLoss()

print("Forget set recreated:", len(forget_indices))

100%|██████████| 9.91M/9.91M [00:00<00:00, 59.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.66MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.32MB/s]

Forget set recreated: 500


In [ ]:

unlearned_model.eval()
unlearned_model.zero_grad()

forget_loss = 0.0
num_forget = 0

for images, labels in forget_loader:
    outputs = unlearned_model(images)
    loss = criterion(outputs, labels)

    forget_loss += loss * images.size(0)
    num_forget += images.size(0)

forget_loss = forget_loss / num_forget
forget_loss.backward()

grad_vector = torch.cat([
    p.grad.detach().flatten()
    for p in unlearned_model.parameters()
    if p.grad is not None
])

print("Forgotten-data loss:", forget_loss.item())
print("Gradient vector size:", grad_vector.numel())
print("Gradient norm:", grad_vector.norm().item())

Forgotten-data loss: 1.0361504554748535
Gradient vector size: 101770
Gradient norm: 0.5467756986618042


In [ ]:

import torch
import torch.nn as nn

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.network(x)

small_model = SmallMLP()

total_params = sum(p.numel() for p in small_model.parameters())

print(small_model)
print("Total parameters:", total_params)

SmallMLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=32, bias=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=10, bias=True)
  )
)
Total parameters: 25450


In [ ]:

import torch
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

small_indices = torch.arange(5000)

small_dataset = torch.utils.data.Subset(
    train_dataset,
    small_indices
)

small_loader = torch.utils.data.DataLoader(
    small_dataset,
    batch_size=128,
    shuffle=True
)

print("Experimental dataset size:", len(small_dataset))

100%|██████████| 9.91M/9.91M [00:00<00:00, 35.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 897kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.08MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.58MB/s]

Experimental dataset size: 5000


In [ ]:

torch.manual_seed(42)

forget_size = 100

forget_indices = torch.randperm(5000)[:forget_size]

retain_indices = torch.tensor([
    i for i in range(5000)
    if i not in set(forget_indices.tolist())
])

forget_dataset = torch.utils.data.Subset(
    small_dataset, forget_indices
)

retain_dataset = torch.utils.data.Subset(
    small_dataset, retain_indices
)

forget_loader = torch.utils.data.DataLoader(
    forget_dataset,
    batch_size=64,
    shuffle=False
)

retain_loader = torch.utils.data.DataLoader(
    retain_dataset,
    batch_size=64,
    shuffle=True
)

print("Forget set:", len(forget_dataset))
print("Retain set:", len(retain_dataset))

Forget set: 100
Retain set: 4900


In [ ]:

import torch.optim as optim

small_full_model = SmallMLP()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    small_full_model.parameters(),
    lr=0.01
)

for epoch in range(3):
    small_full_model.train()
    total_loss = 0

    for images, labels in small_loader:
        optimizer.zero_grad()

        outputs = small_full_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/3, "
        f"Loss: {total_loss / len(small_loader):.4f}"
    )

Epoch 1/3, Loss: 2.2863
Epoch 2/3, Loss: 2.2213
Epoch 3/3, Loss: 2.1425


In [ ]:

import torch.optim as optim

small_full_model = SmallMLP()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    small_full_model.parameters(),
    lr=0.001
)

for epoch in range(5):
    small_full_model.train()
    total_loss = 0

    for images, labels in small_loader:
        optimizer.zero_grad()

        outputs = small_full_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/5, "
        f"Loss: {total_loss / len(small_loader):.4f}"
    )

Epoch 1/5, Loss: 1.7729
Epoch 2/5, Loss: 0.8964
Epoch 3/5, Loss: 0.5787
Epoch 4/5, Loss: 0.4505
Epoch 5/5, Loss: 0.3838


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

torch.save(
    small_full_model.state_dict(),
    "/content/drive/MyDrive/small_full_model.pth"
)

print("Small full model saved.")

Small full model saved.


In [ ]:

retrained_small_model = SmallMLP()

retrain_optimizer = optim.Adam(
    retrained_small_model.parameters(),
    lr=0.001
)

for epoch in range(5):
    retrained_small_model.train()
    total_loss = 0

    for images, labels in retain_loader:
        retrain_optimizer.zero_grad()

        outputs = retrained_small_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        retrain_optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/5, "
        f"Loss: {total_loss / len(retain_loader):.4f}"
    )

Epoch 1/5, Loss: 1.4722
Epoch 2/5, Loss: 0.6060
Epoch 3/5, Loss: 0.4223
Epoch 4/5, Loss: 0.3478
Epoch 5/5, Loss: 0.3086


In [ ]:

torch.save(
    retrained_small_model.state_dict(),
    "/content/drive/MyDrive/retrained_small_model.pth"
)

print("Retrained small model saved.")

Retrained small model saved.


In [ ]:

import torch
import torch.nn as nn

unlearned_small_model = SmallMLP()

unlearned_small_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/small_full_model.pth",
        weights_only=True
    )
)

print("Unlearning model restored from full model.")

Unlearning model restored from full model.


In [ ]:

import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

unlearned_small_model.eval()
unlearned_small_model.zero_grad()

total_loss = 0.0
num_samples = 0

for images, labels in forget_loader:
    outputs = unlearned_small_model(images)
    loss = criterion(outputs, labels)

    total_loss += loss * images.size(0)
    num_samples += images.size(0)

forget_loss = total_loss / num_samples

forget_loss.backward()

grad_vector = torch.cat([
    p.grad.detach().flatten()
    for p in unlearned_small_model.parameters()
    if p.grad is not None
])

print("Forgotten-data loss:", forget_loss.item())
print("Gradient vector size:", grad_vector.numel())
print("Gradient norm:", grad_vector.norm().item())

Forgotten-data loss: 0.36386168003082275
Gradient vector size: 25450
Gradient norm: 0.7921334505081177


In [ ]:

from torch.autograd import grad

unlearned_small_model.zero_grad()

total_loss = 0.0
num_samples = 0

for images, labels in forget_loader:
    outputs = unlearned_small_model(images)
    loss = criterion(outputs, labels)

    total_loss += loss * images.size(0)
    num_samples += images.size(0)

forget_loss = total_loss / num_samples

params = [
    p for p in unlearned_small_model.parameters()
    if p.requires_grad
]

first_grads = grad(
    forget_loss,
    params,
    create_graph=True
)

first_grad_vector = torch.cat([
    g.flatten() for g in first_grads
])

hvp_parts = grad(
    (first_grad_vector * grad_vector).sum(),
    params
)

hvp_vector = torch.cat([
    h.flatten() for h in hvp_parts
])

print("HVP size:", hvp_vector.numel())
print("HVP norm:", hvp_vector.norm().item())

HVP size: 25450
HVP norm: 4.90950345993042


In [ ]:

def compute_hvp(model, loss, params, vector):
    grads = torch.autograd.grad(
        loss,
        params,
        create_graph=True
    )

    grad_vector = torch.cat([
        g.reshape(-1) for g in grads
    ])

    hvp = torch.autograd.grad(
        (grad_vector * vector).sum(),
        params
    )

    return torch.cat([
        h.reshape(-1) for h in hvp
    ]).detach()

In [ ]:

def conjugate_gradient(hvp_fn, b, max_iter=20, tol=1e-6):
    x = torch.zeros_like(b)
    r = b.clone()
    p = r.clone()

    rs_old = torch.dot(r, r)

    for i in range(max_iter):
        Ap = hvp_fn(p)

        denom = torch.dot(p, Ap)

        if torch.abs(denom) < 1e-12:
            print("CG stopped: near-zero denominator.")
            break

        alpha = rs_old / denom

        x = x + alpha * p
        r = r - alpha * Ap

        rs_new = torch.dot(r, r)

        print(f"Iteration {i + 1}: residual = {rs_new.sqrt().item():.6f}")

        if rs_new.sqrt() < tol:
            break

        p = r + (rs_new / rs_old) * p
        rs_old = rs_new

    return x

In [ ]:

unlearned_small_model.eval()

unlearned_small_model.zero_grad()

total_loss = 0.0
num_samples = 0

for images, labels in forget_loader:
    outputs = unlearned_small_model(images)
    loss = criterion(outputs, labels)

    total_loss += loss * images.size(0)
    num_samples += images.size(0)

forget_loss = total_loss / num_samples

params = [
    p for p in unlearned_small_model.parameters()
    if p.requires_grad
]

def hvp_fn(vector):
    return compute_hvp(
        unlearned_small_model,
        forget_loss,
        params,
        vector
    )

print("HVP function ready.")

HVP function ready.


In [ ]:

# First-order practical unlearning

import torch

eta = 0.1

with torch.no_grad():
    for p in unlearned_small_model.parameters():
        if p.grad is not None:
            p.add_(eta * p.grad)

print("First-order unlearning update applied.")

NameError: name 'unlearned_small_model' is not defined

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# Successful First-Order Unlearning Evidence
# Self-contained reconstruction after Colab runtime reset

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Define the same SmallMLP
class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.network(x)

# Recreate the exact final experiment split
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

torch.manual_seed(42)

indices = torch.randperm(5000)
forget_indices = indices[:100]

forget_dataset = Subset(
    train_dataset,
    forget_indices
)

forget_loader = DataLoader(
    forget_dataset,
    batch_size=100,
    shuffle=False
)

criterion = nn.CrossEntropyLoss()

# Restore the trained full model
unlearned_small_model = SmallMLP()

unlearned_small_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/small_full_model.pth",
        weights_only=True
    )
)

unlearned_small_model.train()
unlearned_small_model.zero_grad()

# Calculate forgotten-data loss
total_loss = 0.0
num_samples = 0

for images, labels in forget_loader:
    outputs = unlearned_small_model(images)
    loss = criterion(outputs, labels)

    total_loss += loss * images.size(0)
    num_samples += images.size(0)

forget_loss_before = total_loss / num_samples

# Calculate gradient
forget_loss_before.backward()

# First-order gradient-ascent update
eta = 0.1

with torch.no_grad():
    for p in unlearned_small_model.parameters():
        if p.grad is not None:
            p.add_(eta * p.grad)

# Measure forgotten-data loss after update
unlearned_small_model.eval()

total_loss = 0.0
num_samples = 0

with torch.no_grad():
    for images, labels in forget_loader:
        outputs = unlearned_small_model(images)
        loss = criterion(outputs, labels)

        total_loss += loss * images.size(0)
        num_samples += images.size(0)

forget_loss_after = total_loss / num_samples

print("First-order unlearning successfully executed.")
print(f"Forgotten-data loss before update: {forget_loss_before.item():.4f}")
print(f"Forgotten-data loss after update:  {forget_loss_after.item():.4f}")
print(f"Gradient-ascent step size (eta): {eta}")

First-order unlearning successfully executed.
Forgotten-data loss before update: 0.3639
Forgotten-data loss after update:  0.4429
Gradient-ascent step size (eta): 0.1


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os

for filename in [
    "small_full_model.pth",
    "retrained_small_model.pth",
    "unlearned_small_model.pth"
]:
    path = "/content/drive/MyDrive/" + filename
    print(filename, "→", os.path.exists(path))

small_full_model.pth → True
retrained_small_model.pth → True
unlearned_small_model.pth → True


In [ ]:

import torch
import torch.nn as nn

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.network(x)

full_model = SmallMLP()
unlearned_small_model = SmallMLP()
retrained_model = SmallMLP()

full_model.load_state_dict(
    torch.load('/content/drive/MyDrive/small_full_model.pth',
               map_location='cpu')
)

unlearned_small_model.load_state_dict(
    torch.load('/content/drive/MyDrive/unlearned_small_model.pth',
               map_location='cpu')
)

retrained_model.load_state_dict(
    torch.load('/content/drive/MyDrive/retrained_small_model.pth',
               map_location='cpu')
)

full_model.eval()
unlearned_small_model.eval()
retrained_model.eval()

print("Final experiment models restored successfully.")

Final experiment models restored successfully.


In [ ]:

# ============================================================
# FINAL EVALUATION
# ============================================================

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Recreate the exact 5,000-example experiment split
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

torch.manual_seed(42)
indices = torch.randperm(5000)

forget_indices = indices[:100].tolist()
retain_indices = indices[100:5000].tolist()

forget_dataset = Subset(train_dataset, forget_indices)
retain_dataset = Subset(train_dataset, retain_indices)

forget_loader = DataLoader(forget_dataset, batch_size=100, shuffle=False)
retain_loader = DataLoader(retain_dataset, batch_size=256, shuffle=False)

# Evaluation function
criterion = torch.nn.CrossEntropyLoss()

def evaluate(model, loader):
    total_loss = 0.0
    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, 100 * correct / total


# Evaluate forget and retain sets
models = {
    "Full model": full_model,
    "Unlearned model": unlearned_small_model,
    "Retrained model": retrained_model
}

results = {}

for name, model in models.items():
    forget_loss, forget_acc = evaluate(model, forget_loader)
    retain_loss, retain_acc = evaluate(model, retain_loader)

    results[name] = {
        "forget_loss": forget_loss,
        "forget_accuracy": forget_acc,
        "retain_loss": retain_loss,
        "retain_accuracy": retain_acc
    }


# Test set
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

for name, model in models.items():
    _, test_acc = evaluate(model, test_loader)
    results[name]["test_accuracy"] = test_acc


# Print final results
print("=" * 60)
print("FINAL MACHINE UNLEARNING RESULTS")
print("=" * 60)

for name, r in results.items():
    print(f"\n{name}")
    print(f"  Forget loss:      {r['forget_loss']:.4f}")
    print(f"  Forget accuracy:  {r['forget_accuracy']:.2f}%")
    print(f"  Retain loss:      {r['retain_loss']:.4f}")
    print(f"  Retain accuracy:  {r['retain_accuracy']:.2f}%")
    print(f"  Test accuracy:    {r['test_accuracy']:.2f}%")


# Accuracy changes
forget_drop = (
    results["Full model"]["forget_accuracy"]
    - results["Unlearned model"]["forget_accuracy"]
)

retain_drop = (
    results["Full model"]["retain_accuracy"]
    - results["Unlearned model"]["retain_accuracy"]
)

print("\n" + "=" * 60)
print("UNLEARNING EFFECT")
print("=" * 60)
print(f"Forgotten-data accuracy change: {forget_drop:.2f} percentage points")
print(f"Retained-data accuracy change:  {retain_drop:.2f} percentage points")


# Parameter distances
def parameter_distance(model_a, model_b):
    squared_distance = 0.0
    squared_reference = 0.0

    for p_a, p_b in zip(model_a.parameters(), model_b.parameters()):
        squared_distance += torch.sum((p_a - p_b) ** 2).item()
        squared_reference += torch.sum(p_b ** 2).item()

    return (squared_distance ** 0.5) / (squared_reference ** 0.5)


full_vs_unlearned = parameter_distance(
    full_model,
    unlearned_small_model
)

unlearned_vs_retrained = parameter_distance(
    unlearned_small_model,
    retrained_model
)

print("\n" + "=" * 60)
print("PARAMETER DISTANCE")
print("=" * 60)
print(f"Full vs Unlearned relative distance:     {full_vs_unlearned:.4f}")
print(f"Unlearned vs Retrained relative distance: {unlearned_vs_retrained:.4f}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 64.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.73MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.82MB/s]


FINAL MACHINE UNLEARNING RESULTS

Full model
  Forget loss:      0.3639
  Forget accuracy:  91.00%
  Retain loss:      0.3523
  Retain accuracy:  91.06%
  Test accuracy:    89.07%

Unlearned model
  Forget loss:      0.4429
  Forget accuracy:  87.00%
  Retain loss:      0.3683
  Retain accuracy:  90.45%
  Test accuracy:    88.10%

Retrained model
  Forget loss:      0.3500
  Forget accuracy:  90.00%
  Retain loss:      0.2798
  Retain accuracy:  92.78%
  Test accuracy:    89.88%

UNLEARNING EFFECT
Forgotten-data accuracy change: 4.00 percentage points
Retained-data accuracy change:  0.61 percentage points

PARAMETER DISTANCE
Full vs Unlearned relative distance:     0.0091
Unlearned vs Retrained relative distance: 1.2614


In [ ]:

# Final Experimental Conclusion

The final experiment demonstrates partial machine unlearning using a first-order gradient-ascent update. Accuracy on the forgotten subset decreased by 4.00 percentage points, while accuracy on the retained subset decreased by only 0.61 percentage points.

However, the unlearned model remained substantially different from the retrained model in parameter space. Therefore, this experiment demonstrates partial approximate unlearning rather than exact data erasure.

The Hessian-based formulation explored earlier in the notebook provides the theoretical foundation for a more sophisticated second-order unlearning approach. A Hessian-vector product was successfully demonstrated, while the full iterative second-order solve became computationally impractical on the available CPU. The final experiment therefore uses a first-order baseline.